# 16S Analyses of the Longitudinal Acne Study
## V1-V3 and V4 primer sets comparison

Date created: 12/28/2024

Notebook authors: Yang Chen

Data analysis by: Yang Chen, Tyler Myers, Britta De Pessemier

This notebook plots the following:

- Plot showing abundance of Cutibacterium in each sample with each primer pair (i.e. the axes are V13 vs V4, each point is the amount of Cutibacterium in one sample by each of the primer pairs)
- Venn diagram illustrating the overlap of genera-level taxa detected by both primer sets, alongside those unique to V1-V3 or V4

In [45]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import gemelli
from gemelli.preprocessing import rclr_transformation
from matplotlib_venn import venn2

In [46]:
# Function to load BIOM table, collapse by taxa, sort rows by row sum, remove specified samples, and convert to relative abundance
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    return df


In [47]:
# Read and transform V1-V3 and V4 Genus level absolute abundance table
V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom')
V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom')

In [48]:
# Extract the row corresponding to 'g__Cutibacterium' for V1-V3
V1_V3_cutibacterium_df = V1V3_biom.loc[[' g__Cutibacterium']]

# Rename the row index
V1_V3_cutibacterium_df.index = [' g__Cutibacterium V1-V3']

# Transpose the DataFrame
V1_V3_cutibacterium_df = V1_V3_cutibacterium_df.T

# Display the transformed DataFrame
V1_V3_cutibacterium_df

,g__Cutibacterium V1-V3
LAMI.RD001.D0.C1,2362.0
LAMI.RD001.D0.C2,1808.0
LAMI.RD001.D14.C1,2234.0
LAMI.RD001.D14.C2,1761.0
LAMI.RD001.D28.C1,2367.0
...,...
LAMI.RD319.D14.C1,1900.0
LAMI.RD319.D21.C3,1003.0
LAMI.RD319.D14.C2,2315.0
LAMI.RD319.D9.C1,1230.0


In [49]:
# Extract the row corresponding to 'g__Cutibacterium' for V4
V4_cutibacterium_df = V4_biom.loc[[' g__Cutibacterium']]

# Rename the row index
V4_cutibacterium_df.index = [' g__Cutibacterium V4']

# Transpose the DataFrame
V4_cutibacterium_df = V4_cutibacterium_df.T

# Display the transformed DataFrame
V4_cutibacterium_df

,g__Cutibacterium V4
LAMI.RD001.D0.C1,1.0
LAMI.RD001.D14.C1,0.0
LAMI.RD004.D0.C2,1.0
LAMI.RD001.D0.C2,3.0
LAMI.RD004.D28.C1,0.0
...,...
LAMI.RD319.D16.C2,3.0
LAMI.RD319.D28.C1,0.0
LAMI.RD318.D9.C2,10.0
LAMI.RD319.D28.C2,0.0


In [50]:
# Get ranks for V1-V3
V1_V3_cutibacterium_df['V1-V3'] = V1_V3_cutibacterium_df[' g__Cutibacterium V1-V3'].rank(method='min').astype(int)
V1_V3_cutibacterium_df

,g__Cutibacterium V1-V3,V1-V3
LAMI.RD001.D0.C1,2362.0,31
LAMI.RD001.D0.C2,1808.0,12
LAMI.RD001.D14.C1,2234.0,25
LAMI.RD001.D14.C2,1761.0,11
LAMI.RD001.D28.C1,2367.0,32
...,...,...
LAMI.RD319.D14.C1,1900.0,15
LAMI.RD319.D21.C3,1003.0,4
LAMI.RD319.D14.C2,2315.0,29
LAMI.RD319.D9.C1,1230.0,5


In [51]:
# Get ranks for V4
V4_cutibacterium_df['V4'] = V4_cutibacterium_df[' g__Cutibacterium V4'].rank(method='min').astype(int)
V4_cutibacterium_df

,g__Cutibacterium V4,V4
LAMI.RD001.D0.C1,1.0,12
LAMI.RD001.D14.C1,0.0,1
LAMI.RD004.D0.C2,1.0,12
LAMI.RD001.D0.C2,3.0,42
LAMI.RD004.D28.C1,0.0,1
...,...,...
LAMI.RD319.D16.C2,3.0,42
LAMI.RD319.D28.C1,0.0,1
LAMI.RD318.D9.C2,10.0,79
LAMI.RD319.D28.C2,0.0,1


In [52]:
# Concatenate the two DataFrames along columns, matching on indexes
# Select specific columns
v1_v3_column = V1_V3_cutibacterium_df['V1-V3']  # Adjust column name if needed
v4_column = V4_cutibacterium_df['V4']  # Adjust column name if needed

# Concatenate the selected columns
combined_cutibacterium_df = pd.concat([v1_v3_column, v4_column], axis=1)

# Rename columns for clarity (optional)
combined_cutibacterium_df.columns = ['V1-V3', 'V4']

# Drop rows with NaN values
combined_cutibacterium_df = combined_cutibacterium_df.dropna()

combined_cutibacterium_df

,V1-V3,V4
LAMI.RD001.D0.C1,31.0,12.0
LAMI.RD001.D0.C2,12.0,42.0
LAMI.RD001.D14.C1,25.0,1.0
LAMI.RD001.D14.C2,11.0,1.0
LAMI.RD001.D28.C1,32.0,12.0
...,...,...
LAMI.RD318.D28.C3,77.0,51.0
LAMI.RD319.D14.C1,15.0,12.0
LAMI.RD319.D14.C2,29.0,1.0
LAMI.RD319.D9.C1,5.0,1.0


In [ ]:
# Scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(
    combined_cutibacterium_df.iloc[:, 0], 
    combined_cutibacterium_df.iloc[:, 1], 
    color='#ffa505', 
    edgecolor='black',
    linewidth=0.5,
    alpha=0.7, 
    label='Samples'
)

# Best-fit line
x = combined_cutibacterium_df.iloc[:, 0]
y = combined_cutibacterium_df.iloc[:, 1]
m, b = np.polyfit(x, y, 1)  # Linear regression
correlation = combined_cutibacterium_df.corr().iloc[0, 1]
plt.plot(x, m * x + b, color='darkorange', label=f'Correlation: {correlation:.2f}')

# Title and labels
plt.title('Per-sample correlation between V1-V3 and V4 by Cutibacterium', fontsize=14)
plt.xlabel('Ranked Abundance of Cutibacterium (V1-V3)', fontsize=10)
plt.ylabel('Ranked Abundance of Cutibacterium (V4)', fontsize=10)

# Add correlation coefficient
correlation = combined_cutibacterium_df.corr().iloc[0, 1]

# Grid, legend, and save
plt.grid(True)
plt.legend()
plt.savefig('../Figures/16S_Figures/primer_comparison/Cutbacterium_V1V3_vs_V4_correlation_ranks.png', dpi=600)
plt.show()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_48337/3552077593.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Venn diagram comparing genera-level taxa between V1-V3 and V4

In [54]:
V1V3_biom

,LAMI.RD001.D0.C1,LAMI.RD001.D0.C2,LAMI.RD001.D14.C1,LAMI.RD001.D14.C2,LAMI.RD001.D28.C1,LAMI.RD002.D0.C2,LAMI.RD003.D14.C1,LAMI.RD002.D14.C1,LAMI.RD003.D28.C1,LAMI.RD001.D28.C2,...,LAMI.RD319.D28.C3,LAMI.RD319.D11.C1,LAMI.RD319.D21.C2,LAMI.RD319.D7.C3,LAMI.RD318.D28.C3,LAMI.RD319.D14.C1,LAMI.RD319.D21.C3,LAMI.RD319.D14.C2,LAMI.RD319.D9.C1,LAMI.RD319.D9.C2
g__Cutibacterium,2362.0,1808.0,2234.0,1761.0,2367.0,2900.0,3296.0,3440.0,2367.0,2288.0,...,1686.0,1305.0,2611.0,981.0,3218.0,1900.0,1003.0,2315.0,1230.0,2314.0
g__uncultured,0.0,2.0,2.0,0.0,11.0,0.0,52.0,5.0,127.0,0.0,...,2026.0,2420.0,1123.0,2759.0,538.0,1846.0,2741.0,1410.0,2491.0,1411.0
g__Staphylococcus,94.0,158.0,83.0,137.0,118.0,373.0,95.0,105.0,103.0,204.0,...,10.0,0.0,10.0,21.0,5.0,8.0,11.0,8.0,13.0,8.0
g__Streptococcus,292.0,374.0,240.0,446.0,293.0,159.0,14.0,43.0,543.0,169.0,...,0.0,8.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
g__Corynebacterium,175.0,352.0,492.0,456.0,365.0,39.0,138.0,49.0,31.0,399.0,...,10.0,14.0,7.0,2.0,0.0,6.0,4.0,9.0,3.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g__Methylotenera,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Fusicatenibacter,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Ruminococcus,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Lachnospira,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [55]:
V4_biom

,LAMI.RD001.D0.C1,LAMI.RD001.D14.C1,LAMI.RD004.D0.C2,LAMI.RD001.D0.C2,LAMI.RD004.D28.C1,LAMI.RD011.D0.C2,LAMI.RD001.D14.C2,LAMI.RD004.D28.C2,LAMI.RD017.D14.C2,LAMI.RD011.D14.C1,...,LAMI.RD319.D16.C1,LAMI.RD318.D16.C1,LAMI.RD319.D25.C2,LAMI.RD318.D18.C1,LAMI.RD318.D4.C2,LAMI.RD319.D16.C2,LAMI.RD319.D28.C1,LAMI.RD318.D9.C2,LAMI.RD319.D28.C2,LAMI.RD319.D2.C1
g__Staphylococcus,317.0,454.0,1652.0,349.0,332.0,2132.0,444.0,217.0,1756.0,2063.0,...,58.0,72.0,68.0,83.0,405.0,142.0,42.0,139.0,52.0,120.0
g__uncultured,61.0,29.0,36.0,46.0,16.0,46.0,40.0,32.0,32.0,82.0,...,3562.0,3297.0,3599.0,3291.0,2595.0,3413.0,3525.0,3441.0,3627.0,3509.0
g__Lawsonella,131.0,456.0,95.0,217.0,54.0,291.0,206.0,18.0,636.0,999.0,...,35.0,45.0,54.0,22.0,104.0,57.0,96.0,23.0,54.0,45.0
g__Corynebacterium,275.0,915.0,848.0,509.0,964.0,220.0,546.0,731.0,131.0,102.0,...,16.0,42.0,20.0,25.0,73.0,48.0,38.0,27.0,15.0,27.0
g__Streptococcus,414.0,247.0,164.0,368.0,422.0,153.0,457.0,713.0,33.0,85.0,...,11.0,72.0,1.0,24.0,25.0,2.0,7.0,11.0,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g__Hymenobacter,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Bacteroides,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Absconditabacteriales_(SR1),38.0,0.0,0.0,19.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
g__Sphingomonas,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [56]:
# Function to filter taxa based on prevalence threshold
def filter_by_prevalence(df, threshold=0.01):
    # Calculate prevalence: proportion of samples where the taxon is present
    prevalence = (df > 0).sum(axis=1) / df.shape[1]
    # Keep only taxa that meet the prevalence threshold
    return df[prevalence >= threshold]


In [57]:
# Apply the filtering
filtered_V1V3 = filter_by_prevalence(V1V3_biom, threshold=0.01)
filtered_V4 = filter_by_prevalence(V4_biom, threshold=0.01)

# Get the sets of indices (taxa) from filtered DataFrames
indices_V1V3 = set(filtered_V1V3.index)
indices_V4 = set(filtered_V4.index)

# Taxa unique to V1V3
unique_to_V1V3 = indices_V1V3 - indices_V4

# Taxa unique to V4
unique_to_V4 = indices_V4 - indices_V1V3

# Taxa shared by both
shared_taxa = indices_V1V3 & indices_V4

# Create the Venn diagram
venn = venn2([indices_V1V3, indices_V4], ('V1V3 Taxa (>=1% prevalence)', 'V4 Taxa (>=1% prevalence)'))

# Add a title
plt.title('Venn Diagram of Taxa with >=1% Prevalence')

# Show the plot
plt.savefig('../Figures/16S_Figures/primer_comparison/primer_venn_diagram.png', dpi=600)

In [58]:
# Print the results
print("Unique to V1V3 (>=1% prevalence):")
print(unique_to_V1V3)

print("\nUnique to V4 (>=1% prevalence):")
print(unique_to_V4)

print("\nShared taxa (>=1% prevalence):")
print(shared_taxa)

Unique to V1V3 (>=1% prevalence):
{' g__Pseudoglutamicibacter', ' g__Reyranella', ' g__Blastomonas', ' g__Mycobacterium', ' g__Subdoligranulum', ' g__Microbacterium', ' g__Delftia', ' g__Pseudoalteromonas', ' g__Ferrovibrio', ' g__Rhodopseudomonas', ' g__Hydrogenophaga', ' g__Auricoccus-Abyssicoccus', ' g__Methylobacterium-Methylorubrum', ' g__Janibacter', ' g__Pseudochrobactrum', ' g__Sphingorhabdus', ' g__Ralstonia', ' g__Ramlibacter'}

Unique to V4 (>=1% prevalence):
{' g__Empedobacter', ' g__Stenotrophomonas', ' g__Capnocytophaga', ' g__Aerococcus', ' g__Luteimonas', ' g__Bacteroides', ' g__Aggregatibacter', ' g__Peptostreptococcus', ' g__Enhydrobacter', ' g__Abiotrophia', ' g__Bifidobacterium', ' g__Geobacillus', ' g__Psychrobacter', ' g__Simonsiella', ' g__Agathobacter', ' g__Massilia', ' g__Methylotenera', ' g__Bradyrhizobium', ' g__Peptoniphilus', ' g__Dietzia', ' g__Aeromonas', ' g__Fusobacterium', ' g__Weissella', ' g__Fenollaria', ' g__Pasteurella', ' g__Dolosigranulum', ' g